# Data Catalog MCP Tool Testing

This notebook tests the `search_interaction_events` tool to verify it's working correctly.


## Step 2: Basic Import and Data Loading Test

Testing if we can import the app modules and load the CSV data successfully.

In [1]:
from app import load_interaction_events, search_events, mcp

# Test Step 2: Basic Import and Data Loading Test
try:
    events = load_interaction_events()
    print('✅ Successfully loaded', len(events), 'events')
    print('✅ Sample event:', events[0]['event_name'])
    print('✅ Step 2 PASSED - CSV loading works correctly')
    
    # Show first few events for verification
    print('\n📋 First 5 events:')
    for i, event in enumerate(events[:5]):
        print(f'{i+1}. {event["event_name"]} - {event["stream"]}')
        
except Exception as e:
    print('❌ Step 2 FAILED - Error:', str(e))

✅ Successfully loaded 101 events
✅ Sample event: component_interaction
✅ Step 2 PASSED - CSV loading works correctly

📋 First 5 events:
1. component_interaction - WS1 - Website
2. wayfinder_start - WS1 - Website
3. wayfinder_next - WS1 - Website
4. wayfinder_back - WS1 - Website
5. wayfinder_complete - WS1 - Website


## Step 3A: Exact Match Test

Testing exact event name matching with 100% confidence.

In [2]:
# Step 3A: Exact Match Test
try:
    results = search_events('wayfinder_start')
    print(f'Exact match test: Found {len(results)} results')
    
    if results:
        r = results[0]
        print(f'- Event: {r.event_name}')
        print(f'- Confidence: {r.confidence_score}')
        print(f'- Stream: {r.stream}')
        print(f'- Tool Area: {r.tool_area}')
        
        if r.confidence_score == 100.0:
            print('✅ Step 3A PASSED - Exact match working correctly')
        else:
            print('❌ Step 3A FAILED - Expected 100% confidence for exact match')
    else:
        print('❌ Step 3A FAILED - No results found for exact match')
        
except Exception as e:
    print('❌ Step 3A FAILED - Error:', str(e))

Exact match test: Found 10 results
- Event: wayfinder_start
- Confidence: 100.0
- Stream: WS1 - Website
- Tool Area: Wayfinder
✅ Step 3A PASSED - Exact match working correctly


## Step 3B: Fuzzy Search Test

Testing fuzzy matching with partial event names.

In [3]:
# Step 3B: Fuzzy Search Test
try:
    results = search_events('wayfinder')
    print(f'Fuzzy search test: Found {len(results)} results')
    
    if len(results) >= 3:
        print('\n🔍 Top 3 wayfinder results:')
        for i, r in enumerate(results[:3]):
            print(f'{i+1}. {r.event_name} (score: {r.confidence_score})')
            
        # Check if we have high-confidence wayfinder events
        high_confidence = [r for r in results[:3] if r.confidence_score >= 90]
        if len(high_confidence) >= 2:
            print('✅ Step 3B PASSED - Fuzzy search finding relevant wayfinder events')
        else:
            print('❌ Step 3B FAILED - Expected high confidence wayfinder matches')
    else:
        print('❌ Step 3B FAILED - Expected multiple wayfinder results')
        
except Exception as e:
    print('❌ Step 3B FAILED - Error:', str(e))

Fuzzy search test: Found 10 results

🔍 Top 3 wayfinder results:
1. wayfinder_start (score: 100)
2. wayfinder_next (score: 100)
3. wayfinder_back (score: 100)
✅ Step 3B PASSED - Fuzzy search finding relevant wayfinder events


## Step 3C: Keyword/Phrase Search Test

Testing search within event triggers and descriptions.

In [11]:
# Step 3C: Keyword/Phrase Search Test
try:
    results = search_events('user clicks button')
    print(f'Keyword search test: Found {len(results)} results')
    
    if results:
        print('\n🎯 Click-related events found:')
        for i, r in enumerate(results[:5]):
            print(f'{i+1}. {r.event_name} (score: {r.confidence_score}) - {r.tool_area}')
            
        if len(results) >= 3:
            print('✅ Step 3C PASSED - Keyword search finding click-related events')
        else:
            print('❌ Step 3C FAILED - Expected more click-related results')
    else:
        print('❌ Step 3C FAILED - No results found for click phrase')
        
except Exception as e:
    print('❌ Step 3C FAILED - Error:', str(e))

Keyword search test: Found 10 results

🎯 Click-related events found:
1. wayfinder_start (score: 78) - Wayfinder
2. provider_interaction (score: 78) - Providers
3. eol_complete (score: 78) - Make a referral
4. search_overlay (score: 78) - Site Search
5. search_result_click (score: 78) - Site Search
✅ Step 3C PASSED - Keyword search finding click-related events


## Step 3D: Tool Area Search Test

Testing search by tool/area of the site.

In [12]:
# Step 3D: Tool Area Search Test
try:
    results = search_events('outlets')
    print(f'Tool area search test: Found {len(results)} results')
    
    if results:
        print('\n🏢 Outlet-related events:')
        for i, r in enumerate(results[:5]):
            print(f'{i+1}. {r.event_name} (score: {r.confidence_score})')
            
        # Look for specific outlet events
        outlet_events = [r.event_name for r in results if 'outlet' in r.event_name.lower()]
        if len(outlet_events) >= 2:
            print(f'✅ Step 3D PASSED - Found outlet events: {outlet_events[:3]}')
        else:
            print('❌ Step 3D FAILED - Expected outlet-specific events')
    else:
        print('❌ Step 3D FAILED - No outlet results found')
        
except Exception as e:
    print('❌ Step 3D FAILED - Error:', str(e))

Tool area search test: Found 4 results

🏢 Outlet-related events:
1. outlet_interaction (score: 86)
2. outlet_view (score: 86)
3. outlet_detail_interaction (score: 86)
4. fap_outlet_search (score: 86)
✅ Step 3D PASSED - Found outlet events: ['outlet_interaction', 'outlet_view', 'outlet_detail_interaction']


## Step 4: Edge Case Testing

Testing edge cases like empty queries and limits.

In [13]:
# Step 4: Edge Case Testing
print('🧪 Testing Edge Cases:')

# Empty query test
try:
    results = search_events('')
    print(f'Empty query test: Found {len(results)} results (should be 0)')
    if len(results) == 0:
        print('✅ Empty query handled correctly')
    else:
        print('❌ Empty query should return 0 results')
except Exception as e:
    print('❌ Empty query test failed:', str(e))

# Non-matching query test
try:
    results = search_events('xyz123nonexistent')
    print(f'Non-matching query test: Found {len(results)} results (should be 0)')
    if len(results) == 0:
        print('✅ Non-matching query handled correctly')
    else:
        print('❌ Non-matching query should return 0 results')
except Exception as e:
    print('❌ Non-matching query test failed:', str(e))

# Max results limit test
try:
    results = search_events('a', 5)
    print(f'Limit test: Found {len(results)} results (should be 5 max)')
    if len(results) <= 5:
        print('✅ Result limit working correctly')
    else:
        print('❌ Result limit not working')
except Exception as e:
    print('❌ Limit test failed:', str(e))

🧪 Testing Edge Cases:
Empty query test: Found 0 results (should be 0)
✅ Empty query handled correctly
Non-matching query test: Found 0 results (should be 0)
✅ Non-matching query handled correctly
Limit test: Found 5 results (should be 5 max)
✅ Result limit working correctly


## Step 6: Complete Data Structure Test

Testing that all fields in the EventMatch structure are populated correctly.

In [14]:
# Step 6: Comprehensive Data Structure Test
try:
    results = search_events('outlet_interaction')
    if results:
        r = results[0]
        print('📊 Complete Data Structure Test:')
        print(f'Event Name: {r.event_name}')
        print(f'Trigger: {r.trigger[:100]}...')
        print(f'Confidence: {r.confidence_score}')
        print(f'Stream: {r.stream}')
        print(f'Tool Area: {r.tool_area}')
        print(f'Parameters: {r.parameters[:5]}')  # First 5 params
        print(f'Status: {r.requirements_status}')
        print(f'Link: {r.requirements_link[:60]}...')
        
        # Verify all required fields are populated
        required_fields = [
            r.event_name, r.trigger, r.confidence_score, r.stream,
            r.tool_area, r.requirements_status
        ]
        
        if all(field for field in required_fields):
            print('✅ Step 6 PASSED - All fields populated correctly')
        else:
            print('❌ Step 6 FAILED - Some required fields are missing')
    else:
        print('❌ Step 6 FAILED - No results found for outlet_interaction')
        
except Exception as e:
    print('❌ Step 6 FAILED - Error:', str(e))

📊 Complete Data Structure Test:
Event Name: outlet_interaction
Trigger: This interaction fires across multiple areas of the site - when a user interacts with an outlet card...
Confidence: 100.0
Stream: WS2 - FaP
Tool Area: Outlets
Parameters: ['interaction_type', 'option_selected', 'context', 'section', 'outlet_name']
Status: Ready for Dev
Link: https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA...
✅ Step 6 PASSED - All fields populated correctly


## Final Test Summary

Run this cell to get a summary of all tests.

In [15]:
print('🏁 TESTING COMPLETE')
print('=' * 50)
print('If all tests above show ✅ PASSED, then:')
print('🎉 The search_interaction_events tool has PASSED QA')
print('📝 Ready for documentation in OVERVIEW.md')
print('🚀 Ready for production use!')
print('=' * 50)

🏁 TESTING COMPLETE
If all tests above show ✅ PASSED, then:
🎉 The search_interaction_events tool has PASSED QA
📝 Ready for documentation in OVERVIEW.md
🚀 Ready for production use!
